# 07B — Prepare 159 Main Evaluator Cases + Lock Existing Benchmark Panel — FINAL V2

**Jangan menjalankan benchmark ulang.**

Notebook ini menggunakan hasil benchmark evaluator FINAL V5 yang sudah selesai:

| Rank | Evaluator | Mean Macro-F1 error_type | Std | Self-agreement | Mean joint exact |
|---:|---|---:|---:|---:|---:|
| 1 | Gemini 3.1 Pro Preview | 0.9121 | 0.0231 | 0.9497 | 0.7107 |
| 2 | GPT-5.2 | 0.6936 | 0.0209 | 0.8742 | 0.6101 |
| 3 | Claude Sonnet 4.6 | 0.6631 | 0.0253 | 0.8931 | 0.5849 |

Panel final dikunci menjadi:
1. Gemini 3.1 Pro Preview
2. GPT-5.2
3. Claude Sonnet 4.6

Benchmark V5 menggunakan prompt reference-free dengan hash `d0e3d1938a7dcd05`, provider dipatok, fallback dimatikan, `max_tokens=2048`, dan **temperature tidak dikirim** karena tidak didukung seragam pada seluruh kandidat benchmark.

Notebook ini kemudian menyiapkan 159 output MBR final:
- 53 Zero-Shot
- 53 Fragment-Shot Prompting
- 53 Fine-Tuning LoRA

Reference test, chrF++, BLEU, S3E, dan identitas skenario **tidak masuk prompt evaluator**.

## 1. Dependency audit

In [1]:
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

EXPECTED = {
    "sentence-transformers": "3.4.1",
}

errors = []
for package_name, expected in EXPECTED.items():
    try:
        actual = version(package_name)
    except PackageNotFoundError:
        errors.append(f"{package_name}: NOT INSTALLED")
        continue

    ok = Version(actual) == Version(expected)
    print(
        f"{package_name:24} {actual:12} "
        f"{'OK' if ok else 'MISMATCH'}"
    )

    if not ok:
        errors.append(
            f"{package_name}: actual={actual}, expected={expected}"
        )

if errors:
    raise RuntimeError(
        "Dependency audit gagal:\n- "
        + "\n- ".join(errors)
    )

print("\nDEPENDENCY AUDIT: PASS")

sentence-transformers    3.4.1        OK

DEPENDENCY AUDIT: PASS


## 2. Environment + benchmark-locked panel

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

HF_HOME = Path(
    os.getenv(
        "HF_HOME",
        "/workspace/.hf_home",
    )
).resolve()

EMBEDDING_CACHE = (
    HF_HOME / "sentence_transformers"
).resolve()
EMBEDDING_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_EVAL_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_evaluation"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "prepared"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOCK_PATH = (
    PROJECT_ROOT
    / "Config"
    / "evaluator_panel_locked.json"
)
LOCK_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_CANDIDATES = [
    PROJECT_ROOT
    / "Dataset"
    / "07_dataset_split_final"
    / "train.csv",
    PROJECT_ROOT
    / "Dataset"
    / "train.csv",
]

SCENARIOS = {
    "zero_shot": "Zero-Shot",
    "fsp_locked": "Fragment-Shot Prompting",
    "lora_locked": "Fine-Tuning LoRA",
}

RETRIEVAL_MODEL_ID = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)
RETRIEVAL_TOP_K = 3
SHUFFLE_SEED = 42

BENCHMARK_RUN_ID = "372a575a9fa21f91"
BENCHMARK_PROMPT_HASH = "d0e3d1938a7dcd05"

LOCKED_PANEL = [
    {
        "panel_position": 1,
        "evaluator_key": "E3",
        "display_name": "Gemini 3.1 Pro Preview",
        "model_id": "google/gemini-3.1-pro-preview",
        "developer": "Google",
        "family": "Gemini",
        "access_type": "Proprietary",
        "provider_only": ["google-ai-studio"],
        "mean_macro_f1_error_type": 0.9121,
        "std_macro_f1_error_type": 0.0231,
        "error_pairwise_agreement": 0.9497,
        "mean_joint_exact_match": 0.7107,
        "mean_logical_valid_rate": 1.0,
    },
    {
        "panel_position": 2,
        "evaluator_key": "E1",
        "display_name": "GPT-5.2",
        "model_id": "openai/gpt-5.2",
        "developer": "OpenAI",
        "family": "GPT",
        "access_type": "Proprietary",
        "provider_only": ["openai"],
        "mean_macro_f1_error_type": 0.6936,
        "std_macro_f1_error_type": 0.0209,
        "error_pairwise_agreement": 0.8742,
        "mean_joint_exact_match": 0.6101,
        "mean_logical_valid_rate": 1.0,
    },
    {
        "panel_position": 3,
        "evaluator_key": "E2",
        "display_name": "Claude Sonnet 4.6",
        "model_id": "anthropic/claude-sonnet-4.6",
        "developer": "Anthropic",
        "family": "Claude",
        "access_type": "Proprietary",
        "provider_only": ["anthropic"],
        "mean_macro_f1_error_type": 0.6631,
        "std_macro_f1_error_type": 0.0253,
        "error_pairwise_agreement": 0.8931,
        "mean_joint_exact_match": 0.5849,
        "mean_logical_valid_rate": 0.9937,
    },
]

if len(LOCKED_PANEL) != 3:
    raise RuntimeError("Panel benchmark harus tepat 3 evaluator.")

if len({x["model_id"] for x in LOCKED_PANEL}) != 3:
    raise RuntimeError("Panel benchmark berisi model duplikat.")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("Panel locked :")
display(pd.DataFrame(LOCKED_PANEL))

PROJECT_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi
Panel locked :


,panel_position,evaluator_key,display_name,model_id,developer,family,access_type,provider_only,mean_macro_f1_error_type,std_macro_f1_error_type,error_pairwise_agreement,mean_joint_exact_match,mean_logical_valid_rate
0,1,E3,Gemini 3.1 Pro Preview,google/gemini-3.1-pro-preview,Google,Gemini,Proprietary,[google-ai-studio],0.9121,0.0231,0.9497,0.7107,1.0000
1,2,E1,GPT-5.2,openai/gpt-5.2,OpenAI,GPT,Proprietary,[openai],0.6936,0.0209,0.8742,0.6101,1.0000
2,3,E2,Claude Sonnet 4.6,anthropic/claude-sonnet-4.6,Anthropic,Claude,Proprietary,[anthropic],0.6631,0.0253,0.8931,0.5849,0.9937


## 3. Simpan evaluator_panel_locked.json dari benchmark FINAL V5

In [3]:
lock_payload = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "source":
        "completed_benchmark_evaluator_openrouter_FINAL_V5",
    "benchmark_run_id":
        BENCHMARK_RUN_ID,
    "benchmark_prompt_hash":
        BENCHMARK_PROMPT_HASH,
    "benchmark_reference_free":
        True,
    "benchmark_cases":
        53,
    "benchmark_stage_a_observations":
        318,
    "benchmark_stage_b_additional_calls":
        424,
    "benchmark_total_observations":
        742,
    "selection_rule":
        (
            "Macro-F1 error_type primary; "
            "performance + stability; "
            "no proprietary/open-weight quota"
        ),
    "main_evaluator_protocol": {
        "temperature_sent": False,
        "max_tokens": 2048,
        "provider_pinned": True,
        "provider_fallback": False,
        "shared_evidence": "Top-3 train-only",
        "retrieval_model": RETRIEVAL_MODEL_ID,
    },
    "selected_panel":
        LOCKED_PANEL,
}

LOCK_PATH.write_text(
    json.dumps(
        lock_payload,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("LOCK WRITTEN:", LOCK_PATH)

LOCK WRITTEN: /workspace/10122233_CikitaMeydilvira_Skripsi/Config/evaluator_panel_locked.json


## 4. Load latest COMPLETE 06A–06C tanpa reference

In [4]:
def latest_metric_details(scenario_slug):
    root = FINAL_EVAL_ROOT / scenario_slug

    candidates = sorted(
        root.glob("run_*/final_metric_details.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for path in candidates:
        try:
            df = pd.read_csv(
                path,
                usecols=[
                    "row_id",
                    "source_text",
                    "selected_output",
                ],
                dtype={"row_id": str},
                keep_default_na=False,
                encoding="utf-8-sig",
            )

            if len(df) != 53:
                continue
            if df["row_id"].duplicated().any():
                continue
            if (
                df["selected_output"]
                .astype(str)
                .str.strip()
                .eq("")
                .any()
            ):
                continue

            return path, df

        except Exception:
            continue

    raise FileNotFoundError(
        f"Output 06 COMPLETE tidak ditemukan untuk {scenario_slug}."
    )

frames = []
input_paths = {}

for slug, label in SCENARIOS.items():
    path, df = latest_metric_details(slug)
    input_paths[slug] = str(path)

    local = df.copy()
    local["scenario_slug"] = slug
    local["scenario_label"] = label
    frames.append(local)

    print(f"{label:28} -> {path}")

private_cases = pd.concat(
    frames,
    ignore_index=True,
)

if len(private_cases) != 159:
    raise RuntimeError(
        f"Harus 159 output MBR, ditemukan {len(private_cases)}."
    )

counts = (
    private_cases
    .groupby("scenario_slug")
    .size()
    .to_dict()
)

expected_counts = {
    "zero_shot": 53,
    "fsp_locked": 53,
    "lora_locked": 53,
}

if counts != expected_counts:
    raise RuntimeError(
        f"Distribusi skenario salah: {counts}"
    )

source_pivot = private_cases.pivot(
    index="row_id",
    columns="scenario_slug",
    values="source_text",
)

if len(source_pivot) != 53:
    raise RuntimeError("Harus ada 53 row_id test unik.")

if not (source_pivot.nunique(axis=1) == 1).all():
    raise RuntimeError(
        "source_text berbeda antar skenario."
    )

print("159 OUTPUT MBR INPUT: PASS")

Zero-Shot                    -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/zero_shot/run_1ef71f2034f92d52/final_metric_details.csv
Fragment-Shot Prompting      -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065/final_metric_details.csv
Fine-Tuning LoRA             -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/lora_locked/run_20260813T180837Z/final_metric_details.csv
159 OUTPUT MBR INPUT: PASS


## 5. Train-only shared evidence Top-3

In [5]:
def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(
        "train.csv tidak ditemukan:\n"
        + "\n".join(str(p) for p in paths)
    )

TRAIN_PATH = first_existing(TRAIN_CANDIDATES)

train_df = pd.read_csv(
    TRAIN_PATH,
    usecols=[
        "row_id",
        "conversation_no",
        "turn_order",
        "duri_text",
        "indonesia_text",
    ],
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

if len(train_df) != 417:
    raise RuntimeError(
        f"Train final harus 417, ditemukan {len(train_df)}."
    )

queries = (
    private_cases[
        ["row_id", "source_text"]
    ]
    .drop_duplicates("row_id")
    .sort_values("row_id", kind="stable")
    .reset_index(drop=True)
)

if len(queries) != 53:
    raise RuntimeError("Unique test source harus 53.")

def norm_text(x):
    return re.sub(
        r"\s+",
        " ",
        str(x or "").strip().casefold(),
    )

retriever = SentenceTransformer(
    RETRIEVAL_MODEL_ID,
    device="cpu",
    cache_folder=str(EMBEDDING_CACHE),
)

train_emb = retriever.encode(
    train_df["duri_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

query_emb = retriever.encode(
    queries["source_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

evidence_rows = []
evidence_map = {}

for i, q in queries.iterrows():
    scores = train_emb @ query_emb[i]
    order = np.argsort(-scores)

    source_norm = norm_text(q["source_text"])
    selected = []

    for j in order:
        tr = train_df.iloc[int(j)]

        if norm_text(tr["duri_text"]) == source_norm:
            continue

        selected.append({
            "rank": len(selected) + 1,
            "train_row_id": str(tr["row_id"]),
            "duri_text": str(tr["duri_text"]).strip(),
            "indonesia_text": str(tr["indonesia_text"]).strip(),
            "similarity": float(scores[int(j)]),
        })

        if len(selected) == RETRIEVAL_TOP_K:
            break

    if len(selected) != RETRIEVAL_TOP_K:
        raise RuntimeError(
            f"Evidence kurang untuk row_id={q['row_id']}."
        )

    blocks = []

    for item in selected:
        blocks.append(
            f"Contoh {item['rank']}:\n"
            f"- Duri: {item['duri_text']}\n"
            f"- Indonesia: {item['indonesia_text']}"
        )

        evidence_rows.append({
            "row_id": str(q["row_id"]),
            **item,
        })

    evidence_map[str(q["row_id"])] = "\n\n".join(blocks)

evidence_df = pd.DataFrame(evidence_rows)

if len(evidence_df) != 53 * 3:
    raise RuntimeError(
        f"Evidence rows harus 159, ditemukan {len(evidence_df)}."
    )

print("SHARED EVIDENCE: PASS")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SHARED EVIDENCE: PASS


## 6. Blind + deterministic shuffle 159 kasus

In [6]:
prepared = private_cases.copy()
prepared["shared_evidence"] = (
    prepared["row_id"].map(evidence_map)
)

scenario_order = {
    "zero_shot": 1,
    "fsp_locked": 2,
    "lora_locked": 3,
}

prepared["_scenario_order"] = (
    prepared["scenario_slug"].map(scenario_order)
)

prepared = (
    prepared
    .sort_values(
        ["row_id", "_scenario_order"],
        kind="stable",
    )
    .reset_index(drop=True)
)

rng = np.random.default_rng(SHUFFLE_SEED)
prepared = (
    prepared
    .iloc[rng.permutation(len(prepared))]
    .reset_index(drop=True)
)

prepared["eval_case_id"] = [
    f"EVAL{i:03d}"
    for i in range(1, len(prepared) + 1)
]

blind_df = prepared[
    [
        "eval_case_id",
        "source_text",
        "selected_output",
        "shared_evidence",
    ]
].rename(
    columns={
        "selected_output": "candidate_text",
    }
)

private_mapping_df = prepared[
    [
        "eval_case_id",
        "row_id",
        "scenario_slug",
        "scenario_label",
    ]
].copy()

forbidden_blind = {
    "row_id",
    "scenario_slug",
    "scenario_label",
    "reference_text",
    "indonesia_text",
    "sentence_bleu_13a",
    "sentence_chrfpp",
    "s3e_value",
    "selected_utility",
}

leaked = forbidden_blind & set(blind_df.columns)

if leaked:
    raise RuntimeError(
        f"Blind input bocor: {sorted(leaked)}"
    )

if len(blind_df) != 159:
    raise RuntimeError("Blind evaluator input bukan 159.")

if blind_df["eval_case_id"].duplicated().any():
    raise RuntimeError("eval_case_id duplikat.")

BLIND_PATH = (
    OUTPUT_DIR
    / "main_evaluator_blind_159.csv"
)
MAPPING_PATH = (
    OUTPUT_DIR
    / "main_evaluator_private_mapping.csv"
)
EVIDENCE_PATH = (
    OUTPUT_DIR
    / "main_evaluator_shared_evidence_detail.csv"
)
MANIFEST_PATH = (
    OUTPUT_DIR
    / "main_evaluator_preparation_manifest.json"
)

blind_df.to_csv(
    BLIND_PATH,
    index=False,
    encoding="utf-8-sig",
)

private_mapping_df.to_csv(
    MAPPING_PATH,
    index=False,
    encoding="utf-8-sig",
)

evidence_df.to_csv(
    EVIDENCE_PATH,
    index=False,
    encoding="utf-8-sig",
)

manifest = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "cases": 159,
    "scenarios": 3,
    "rows_per_scenario": 53,
    "shuffle_seed": SHUFFLE_SEED,
    "reference_read": False,
    "scenario_identity_in_blind_input": False,
    "automatic_metrics_in_blind_input": False,
    "benchmark_run_id": BENCHMARK_RUN_ID,
    "benchmark_prompt_hash": BENCHMARK_PROMPT_HASH,
    "panel_lock_path": str(LOCK_PATH),
    "shared_evidence": {
        "source": "train_only",
        "train_rows": 417,
        "top_k": RETRIEVAL_TOP_K,
        "embedding_model": RETRIEVAL_MODEL_ID,
        "same_evidence_for_same_source": True,
        "exact_source_match_excluded": True,
        "test_reference_used_for_retrieval": False,
    },
    "input_paths": input_paths,
    "train_path": str(TRAIN_PATH),
    "outputs": {
        "blind_input": str(BLIND_PATH),
        "private_mapping": str(MAPPING_PATH),
        "evidence_detail": str(EVIDENCE_PATH),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

display(blind_df.head(8))
print("BLIND  :", BLIND_PATH)
print("MAPPING:", MAPPING_PATH)
print("LOCK   :", LOCK_PATH)

,eval_case_id,source_text,candidate_text,shared_evidence
0,EVAL001,Matumbara joq tedong malillin?,Bagaimana cara memasak kerbau?,Contoh 1:\n- Duri: Ia paleq ke susii tuq waqdi...
1,EVAL002,Teqda. Ia to tanduk tedong malillin tuo lako b...,Tidak. Tanduk kerbau yang sudah besar akan kel...,Contoh 1:\n- Duri: Ia paleq ke susii tuq waqdi...
2,EVAL003,Pirang taunmira to muane?,Pirang taunmira to muane?,Contoh 1:\n- Duri: Apara disanga mangpaendek?\...
3,EVAL004,"Teqda kitaq kuissenni, mbai den jao Juppandang.","Saya tidak tahu, mungkin ada di situ Juppandang.",Contoh 1:\n- Duri: La kulianaq ke denni dipuon...
4,EVAL005,Umbori joq too?,"Saya tidak tahu, mungkin disekitar sini saja d...",Contoh 1:\n- Duri: Apara joq noni nenaq?\n- In...
5,EVAL006,Piramira isinna?,Piramira isinna?,Contoh 1:\n- Duri: Piramirikiq sainna melajah ...
6,EVAL007,Apara dikabuaq doke?,Kamu apa kabar?,Contoh 1:\n- Duri: Umbora tapangngujui?\n- Ind...
7,EVAL008,Denmo.,Ada.,Contoh 1:\n- Duri: Denmo tonna teq namanyumpun...


BLIND  : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/prepared/main_evaluator_blind_159.csv
MAPPING: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/prepared/main_evaluator_private_mapping.csv
LOCK   : /workspace/10122233_CikitaMeydilvira_Skripsi/Config/evaluator_panel_locked.json


## 7. Final audit

In [7]:
checks = {
    "blind_159":
        len(blind_df) == 159,
    "mapping_159":
        len(private_mapping_df) == 159,
    "three_scenarios_private":
        private_mapping_df["scenario_slug"].nunique() == 3,
    "53_each_private":
        bool(
            (
                private_mapping_df
                .groupby("scenario_slug")
                .size()
                == 53
            ).all()
        ),
    "reference_not_read":
        True,
    "scenario_hidden_from_blind":
        not bool(
            {"scenario_slug", "scenario_label", "row_id"}
            & set(blind_df.columns)
        ),
    "lock_top3":
        len(LOCKED_PANEL) == 3,
    "lock_exists":
        LOCK_PATH.exists(),
    "blind_exists":
        BLIND_PATH.exists(),
    "manifest_exists":
        MANIFEST_PATH.exists(),
}

failed = [
    k for k, v in checks.items()
    if not bool(v)
]

display(
    pd.DataFrame(
        [
            {"check": k, "ok": bool(v)}
            for k, v in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL PREP AUDIT GAGAL: {failed}"
    )

print("\n" + "=" * 78)
print("BENCHMARK PANEL LOCK + 159 MAIN CASE PREPARATION: COMPLETE")
print("=" * 78)
print("Top-3:")
for x in LOCKED_PANEL:
    print(
        x["panel_position"],
        x["display_name"],
        "->",
        x["model_id"],
    )

,check,ok
0,blind_159,True
1,mapping_159,True
2,three_scenarios_private,True
3,53_each_private,True
4,reference_not_read,True
5,scenario_hidden_from_blind,True
6,lock_top3,True
7,lock_exists,True
8,blind_exists,True
9,manifest_exists,True



BENCHMARK PANEL LOCK + 159 MAIN CASE PREPARATION: COMPLETE
Top-3:
1 Gemini 3.1 Pro Preview -> google/gemini-3.1-pro-preview
2 GPT-5.2 -> openai/gpt-5.2
3 Claude Sonnet 4.6 -> anthropic/claude-sonnet-4.6
